# 05 · Building similarity and learned models (Phase 6)

What surroundings do buildings sit in, and which non-built cells look like them?

1. reference set: where the reference buildings are (P6.2);
2. random forests: the change RF (2018 surroundings of cells that became built in 2020/21) and the state RF (built vs non-built), spatial-block CV (P6.5);
3. held-out building check: do surroundings find buildings in unseen 2 km blocks (P6.6);
4. comparison of the scores.

*Example building queries (P6.3) and the aggregate kNN similarity (P6.4) are added here with Harsh's `query.py` / `aggregate.py`.*

Build: `python -m src.similarity.reference`, `python -m src.similarity.rf_model`, `python -m src.similarity.holdout`, `python -m src.viz.static_maps`, `python -m src.viz.charts`. Checked by `scripts/verify_similarity_models.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from src.config import load_config

cfg = load_config()
MET = Path(cfg.paths["metrics"])
FIG = Path(cfg.paths["figures"])


def show(name, width=760):
    display(Image(filename=str(FIG / f"{name}.png"), width=width))

## 1. Reference set

Cells with an OSM building, plus ESRI built cells where OSM has none (OSM maps 62–75 % of built cells in the north but ~1 % in the south in 2018).

In [ ]:
from src.similarity.reference import reference_path

rows = []
for run in ("validation", "final"):
    ref = pd.read_parquet(reference_path(cfg, run))
    rows.append({"run": run, "reference cells": len(ref), **ref["source"].value_counts().to_dict(),
                 "OSM buildings": int(ref["n_buildings"].sum())})
pd.DataFrame(rows).set_index("run")

In [ ]:
from src.similarity.reference import buildings

b = buildings(cfg, "validation")
b["type_group"].value_counts(normalize=True).round(4).to_frame("share of 2018 OSM buildings")

Almost every OSM building is untyped (`building=yes`), so similarity per building type (P6.7) is not possible on this data.

## 2. Random forests

In [ ]:
sim = json.loads((MET / "similarity.json").read_text())
pd.DataFrame(
    {
        m: {
            "training label": sim[m]["training"]["label"],
            "training cells": sim[m]["training"]["n_cells"],
            "positives": sim[m]["training"]["n_positive"],
            "best params": sim[m]["best_params"],
            "spatial-CV AUC (training label)": round(sim[m]["spatial_cv_auc"]["mean"], 3),
            "AUC on grew, out of fold": round(sim[m]["temporal_preview"]["auc_grew_oof"], 3),
            "distance-to-built AUC": round(sim[m]["temporal_preview"]["auc_grew_dist_to_built"], 3),
        }
        for m in ("rf_change", "rf_state")
    }
)

**Reading it.** The state RF separates built from non-built almost perfectly (0.998), mostly through distance to built-up, but that skill doesn't carry over to *where growth happens next* (0.702, below the plain distance-to-built baseline). The change RF learns from cells that actually changed and beats the baseline (0.779 vs 0.763). Scored on its own training cells it would claim 0.997, which is why validation scores are out of fold (D11).

In [ ]:
show("chart_roc", 520)

In [ ]:
show("chart_rf_importance")

Built-up share within 500 m dominates; trees and crops nearby and distance to built-up follow. Terrain and water barely matter once the built-up context is known.

In [ ]:
show("map_score_rf_change_validation", 600)
show("map_score_rf_change_final", 600)

## 3. Held-out building check

In [ ]:
h = sim["holdout"]["validation"]["scorers"]
pd.DataFrame(h).T[["auc", "median_percentile", "best_possible_median", "share_top10", "n_held_out"]].round(3)

Medians are capped near 70 because reference cells are ~60 % of the non-excluded cells, so AUC is the fair measure. Keeping both ring scales helps (kNN 0.895 vs 0.871 with 500 m rings only), which settles the question from notebook 03.

In [ ]:
show("chart_holdout")

## 4. Comparison

The aggregate similarity score (P6.4) joins this comparison once it exists; Phase 7 combines it with `rf_change`.

In [ ]:
show("map_score_rf_state_final", 600)